# CSCI 3151 — M08 mini‑lab: Maximum likelihood in a click‑through experiment

This notebook is for the **"Do"** part of M08 (Maximum likelihood estimation: principle & examples).

**Goals (10–15 minutes).**

- Practise going from a *data‑generating story* to a simple probabilistic model.
- Implement the **Bernoulli MLE** in Python.
- Plot and interpret **log‑likelihood curves** for two subject lines.
- Produce a short **decision snapshot** you can upload or discuss.

> _Science + engineering mindset:_ keep track of the DGP, units, splits, and where your estimate might fail.


## 0. Assumptions, decision context, and ethics

We imagine a non‑profit organization testing a new email subject line.

- Each row in our dataset is **one email impression** (one person receiving one email in a specific campaign).
- Column `group` is either `"control"` (old subject line) or `"new_subject"`.
- Column `clicked` is `1` if the recipient clicked the donation link within **48 hours**, else `0`.

**Decision maker.**  
A data‑savvy campaign manager who must decide whether to **ship the new subject line next week**.

**Costs of errors.**

- Treating a bad subject line as good → wasted staff time, donor fatigue, fewer long‑term donations.
- Treating a good subject line as bad → missed short‑term donations for the same costs.

**Ethics note.**

Even in this toy setting, we should be careful about:

- **Fairness.** Click behaviour can differ across age, income, language, etc. Training only on a narrow slice of users can produce misleading likelihoods.
- **Privacy.** Real systems should avoid keeping personally identifiable information (PII) longer than needed and be transparent about tracking.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make plots a bit bigger by default
plt.rcParams["figure.figsize"] = (6, 4)

rng = np.random.default_rng(3151)


## 1. Create and inspect a tiny click‑through dataset

We'll simulate a small experiment:

- `n_control = 200` people see the **old** subject line.
- `n_new = 200` people see the **new** subject line.
- True click probabilities (hidden from us) are:

  - `p_control_true = 0.10`  (10%),
  - `p_new_true = 0.13`      (13%).

We will **pretend** this synthetic dataset came from a real A/B test.


In [2]:
n_control, n_new = 200, 200
p_control_true, p_new_true = 0.10, 0.13

control = rng.binomial(1, p_control_true, size=n_control)
new = rng.binomial(1, p_new_true, size=n_new)

df = pd.DataFrame(
    {
        "group": ["control"] * n_control + ["new_subject"] * n_new,
        "clicked": np.concatenate([control, new]),  # 0/1 within 48h
    }
)

df.head()


,group,clicked
0,control,0
1,control,0
2,control,0
3,control,0
4,control,1


In [3]:
print("Number of rows:", len(df))
print("\nGroup counts:")
print(df["group"].value_counts())

print("\nOverall click-through rate (CTR):")
overall_ctr = df["clicked"].mean()
print(f"{overall_ctr:.3f} (fraction of 1s)")


Number of rows: 400

Group counts:
group
control        200
new_subject    200
Name: count, dtype: int64

Overall click-through rate (CTR):
0.138 (fraction of 1s)


## 2. Task 1 — Implement the Bernoulli MLE

For Bernoulli data (0/1 clicks), the likelihood for parameter $p$ given $k$ clicks out of $n$ emails is

\[
L(p; \mathcal{D}) = p^{k} (1-p)^{n-k},
\]

and the MLE is

\[
\hat p_{\text{MLE}} = \frac{k}{n}.
\]

Fill in the function below so that it returns the MLE. The `assert` statements should pass once you're done.


In [4]:
def bernoulli_mle(k: int, n: int) -> float:
    """Maximum likelihood estimator for the Bernoulli probability p.

    Parameters
    ----------
    k : int
        Number of "successes" (clicked = 1).
    n : int
        Number of trials (emails sent).

    Returns
    -------
    float
        MLE p_hat = k / n.
    """
    # TODO: replace the line below with the correct formula
    raise NotImplementedError("Implement p_hat = k / n")


# Quick self-checks (uncomment after you implement the function)
try:
    assert np.isclose(bernoulli_mle(7, 10), 0.7)
    assert np.isclose(bernoulli_mle(3, 4), 0.75)
    print("bernoulli_mle seems to be working ✅")
except NotImplementedError:
    print("TODO: implement bernoulli_mle(k, n) above.")
except AssertionError:
    print("Something is off — check your formula for p_hat.")


TODO: implement bernoulli_mle(k, n) above.


## 3. Task 2 — Summarize click rates by group

Now compute, for each group, how many emails were sent and how many were clicked.

Use a `groupby` aggregation to produce a table with columns:

- `n` – number of emails,
- `k` – number of clicks,
- `p_hat` – MLE estimate of click probability (CTR).

(Tip: once `bernoulli_mle` works, you can re‑use it here.)


In [5]:
group_summary = (
    df
      .groupby("group")["clicked"]
      .agg(n="count", k="sum")
)

# TODO: add a column 'p_hat' with the MLE per group
# Option 1: vectorize: group_summary["p_hat"] = group_summary["k"] / group_summary["n"]
# Option 2: apply a small helper using bernoulli_mle.

group_summary


,n,k
group,,
control,200,26
new_subject,200,29


In [6]:
# Quick sanity check (run after you define the 'p_hat' column)
if "p_hat" in group_summary.columns:
    # CTR must lie within [0, 1]
    assert (group_summary["p_hat"].between(0, 1)).all()
    print("p_hat column exists and is within [0, 1] ✅")
else:
    print("TODO: add a 'p_hat' column to group_summary.")


TODO: add a 'p_hat' column to group_summary.


### Reflection

Use the table above to answer:

1. Which group has the larger estimated click probability $\hat p$?
2. By how many **percentage points** does it differ from the other group?
3. Does that difference feel practically important in this campaign?

(You don't have to write full sentences here — short notes are fine.)


## 4. Task 3 — Plot log‑likelihood curves

The log‑likelihood for Bernoulli data is

\[
\ell(p) = k \log p + (n-k)\log(1-p).
\]

We want to see how sharply the data prefer one value of $p$ over others for each group.

1. Implement `log_likelihood_bernoulli` below.
2. Evaluate it on a grid of $p$ values for each group.
3. Make a plot showing both curves and mark each group’s $\hat p$.


In [7]:
def log_likelihood_bernoulli(k: int, n: int, p: np.ndarray) -> np.ndarray:
    """Log-likelihood for k successes out of n Bernoulli(p) trials.

    Parameters
    ----------
    k : int
        Number of "successes".
    n : int
        Number of trials.
    p : np.ndarray
        Grid of candidate probabilities (each between 0 and 1).

    Returns
    -------
    np.ndarray
        Log-likelihood values with the same shape as `p`.
    """
    # TODO: implement using the formula ℓ(p) = k log p + (n-k) log(1-p)
    raise NotImplementedError("Implement the Bernoulli log-likelihood.")


# Prepare a probability grid
p_grid = np.linspace(0.01, 0.35, 200)

k_control = int(group_summary.loc["control", "k"])
n_control = int(group_summary.loc["control", "n"])
k_new = int(group_summary.loc["new_subject", "k"])
n_new = int(group_summary.loc["new_subject", "n"])

try:
    ll_control = log_likelihood_bernoulli(k_control, n_control, p_grid)
    ll_new = log_likelihood_bernoulli(k_new, n_new, p_grid)

    plt.figure()
    plt.plot(p_grid, ll_control, label="control")
    plt.plot(p_grid, ll_new, label="new_subject")
    if "p_hat" in group_summary.columns:
        plt.axvline(group_summary.loc["control", "p_hat"], linestyle="--")
        plt.axvline(group_summary.loc["new_subject", "p_hat"], linestyle="--")
    plt.xlabel("p (click probability)")
    plt.ylabel("log-likelihood ℓ(p)")
    plt.legend()
    plt.title("Log-likelihood curves by group")
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("TODO: implement log_likelihood_bernoulli(k, n, p) first.")


TODO: implement log_likelihood_bernoulli(k, n, p) first.


### Reflection

Look at your plot and discuss (with a neighbour if possible):

- Is the likelihood for the new subject line **much sharper** or only slightly different?
- How much overlap is there between plausible $p$ values for each group?
- What would happen to the curves if we ran the experiment with **4× more emails**?


## 5. Decision snapshot (for upload / discussion)

In the cell below, write a short "decision snapshot" you could share with a busy stakeholder.

Aim for **5–7 bullet points** covering:

- **Problem & metric** (what decision, what evaluation metric).
- **Baseline & estimate** (old vs new CTR, maybe with rough uncertainty language).
- **Decision** (ship / wait / run a bigger test).
- **Next test** you would run to strengthen the evidence.
- **One risk** (fairness / privacy / robustness) and how you might mitigate it.

Keep it concise and concrete. Imagine they only have **30 seconds** to read it.


<!-- TODO: Replace the bullets below with your own decision snapshot. -->

- Problem & metric: …
- Baseline & estimate: …
- Decision: …
- Next test: …
- Risk & mitigation: …
